# Introduction

This is a self-guided step-by-step Jupyter notebook guide that will show you how to run Python scripts saved on the
```/scripts``` directory of the [precog-esgf-intake](https://github.com/precog-ocean/precog-esgf-intake) repository.
The scripts will be run from within this Jupyter notebook for illustrative purposes.

To run each ```.py``` script, simply run each cell in this notebook and follow the in-cell prompts as if you were working on any ```Terminal prompt```.

You can also run equivalent commands on any ```Terminal prompt``` with slightly different semantics (shown below).


## Step 1. Check / Activate Python Environment
Type the following command to activate the virtual environment from within your Jupyter notebook server:


In [ ]:
%%bash
source .venv/bin/activate


The command above is similar to running the following on a ```Terminal prompt```:
```bash
source .venv/bin/activate
```


## Step 2. Create directory to save search results

Create a directory named ```test_search``` under your Desktop.


In [ ]:
%%bash
mkdir -p ~/Desktop/test_search


PS. This is equivalent to running ```> mkdir ~/Desktop/test_search``` on a ```Terminal prompt```


Run the next cell to display an example TOML layout that can be adapted to your own search criteria.


## Step 2.1. Edit the TOML search configuration

Before running the ESGF catalogue sweep, edit the user-facing TOML configuration file so that it matches your intended
ESGF search. This makes the workflow more reproducible than changing hard-coded Python dictionaries inside the script.

The TOML file can be used to predefine search facets such as project, activity, experiment, frequency, variables, and grid labels. A typical configuration might include entries such as `project = ['CMIP6']`, `experiment_id = ['piControl', 'historical']`, `frequency = ['mon']`, and `grid_label = ['gn', 'gr']`.

If `variable_id` is intentionally left empty in the TOML file, the catalogue script can still fall back to an interactive prompt so that you can type the variables at run time. This is useful when the rest of the search criteria should remain fixed but the variable list changes between runs.

In [ ]:
example_toml = '''
[paths]
download_path = "~/Desktop/test_search"

[esgf]
project = ["CMIP6"]
activity_drs = ["CMIP", "ScenarioMIP"]
experiment_id = ["piControl", "historical"]
frequency = ["mon"]
variable_id = ["expc", "epc100"]
grid_label = ["gn", "gr"]
'''
print(example_toml)


The exact filename of the TOML file can follow your package implementation, for example `search_criteria.toml` or a user-supplied config passed on the command line. The key point is that the search criteria are now externalised and can be edited before Step 3.


## Step 3. ESGF Catalogue Search for ESM outputs of interest

Now run the next cell in the notebook to execute `intake_CatalogueSearch.py` and follow the CLI in-prompt instructions.
When asked to provide `variable_ids`, insert `['expc', 'epc100']`.


In [ ]:
import os.path
%run scripts/intake_CatalogueSearch.py


PS. This is equivalent to running the command `> python scripts/intake_CatalogueSearch.py` on a `Terminal prompt`.

### User prompts

The catalogue sweep supports two related screening modes:

- **Single-branch mode** keeps one internally consistent pairing of `piControl` and `historical` files for each shortlisted model and grid.
- **Ensemble-branch mode** keeps one validated `piControl` anchor branch and then retains all compatible `historical` ensemble members that share the same model and grid and pass the variable-availability and continuity checks.

Both modes first test whether all requested variables are available for `piControl` and `historical`, and whether they coexist on at least one common grid label such as `gn` or `gr`.

Type `single` to enter the single-branch mode.

Then in the next prompt type `yes` to perform a union search and insert `['expc', 'epc100']`.

### Search Logs

You'll see the search log printed to screen, which is also saved for future reference under your
`test_search` directory.

For instance, under 'single-branch' mode the log shows that complete PI and
Historical runs for both  `epc100` and `expc` where found for the following CMIP6 models:
```
- GFDL-ESM4
- GISS-E2-1-G
- IPSL-CM6A-LR
- MPI-ESM-1-2-HAM
- MPI-ESM1-2-HR
- MPI-ESM1-2-LR
- UKESM1-0-LL
```

The log file produced also shows readable grid-level diagnostics. For example, a model may have `expc` available on
`gn` for `piControl` but fail the corresponding `historical` test on that same grid, in which case that grid is
rejected from downstream screening. For example, model `GISS-E2-1-G-CC` has `expc` available in a native grid (i.e., `gn`) but is lacking `expc` outputs for the Historical run in the native grid.

```
No complete set of variables for model GISS-E2-1-G-CC for variable ['expc', 'epc100'] in either grid. Test returned:
INFO - ####################################################################################################
INFO - grid_label       var_test   variable_ids  has_all_variables        run          model
INFO -         gn  [True, False] [expc, epc100]              False  piControl GISS-E2-1-G-CC
INFO -         gr [False, False] [expc, epc100]              False  piControl GISS-E2-1-G-CC
INFO -         gn  [False, True] [expc, epc100]              False historical GISS-E2-1-G-CC
INFO -         gr [False, False] [expc, epc100]              False historical GISS-E2-1-G-CC
INFO - ####################################################################################################
```

### Files produced

Now look at the `path` you indicated (i.e., `test_search/`) and inspect the files created. You should have the
following:

- `ESGF_search_<datetime_stamp>.xlsx` ==> a dataframe with the raw search results returned from ESGF nodes.
- `ESGF_search_<datetime_stamp>_<varstamp>.log` ==> a text log containing the results of the ESGF sweep together with variable-completeness, grid-consistency, and continuity checks.
- `DF_Downloadable_<varstamp>.xlsx` ==> a filtered dataframe of downloadable rows that passed the screening tests.
- `DF_Downloadable_Ensemble_<varstamp>.xlsx`, if you had run the search under `ensemble` mode.

### Step 3.1

The dataframe `DF_Downloadable_expc_epc100.xlsx` contains the rows that survived the catalogue screening.

You can add custom filtering before passing `DF_Downloadable_<varstamp>.xlsx` to the downloader script
`intake_OceanVarsDL.py`, which we will be using in **Step 4**.

For instance, you could create a second dataframe containing only `UKESM1-0-LL` entries by running the following subprogram:

In [ ]:
import pandas as pd
from pathlib import Path

def filter_model(path, sheet_name, model):
    # Read the sheet into a DataFrame
    df = pd.read_excel(path, sheet_name=0)  # sheet_name=0 = first sheet
    fname = path.name.split('.')[0] + '_' + model + '.xlsx'
    # Keep only the model you want
    df_model = df[df['source_id'] == model]
    # Export filtered file
    df_model.to_excel(os.path.join(path.parent, fname), sheet_name=sheet_name, index=False)
    print(f"File {path} has been trimmed.")

if __name__ == "__main__":
    # Path to your file
    excel_path = input("Now either drag onto terminal or type path to Dataframe with the Filtered ESGF search results:")
    excel_path = Path(excel_path.strip(" "))  # strip needed as dragging onto terminal adds a trailing 'space'
    filter_model(excel_path, sheet_name='Sheet1', model='UKESM1-0-LL')


For the sake of illustration, you can use the same logic from the subprogram above so that it will delete all rows but the top 5 on this `UKESM1-0-LL` dataframe. Otherwise, it may prompt you to accept a large download to your disk once you run the program that will fetch data in **Step 4**.

You can trim the dataframe by running this subprogram below:


In [ ]:
import pandas as pd
from pathlib import Path

def keep_top_five_rows(path: Path, sheet_name):
    # Read the sheet into a DataFrame
    df = pd.read_excel(path, sheet_name=0)  # sheet_name=0 = first sheet
    # Keep only the first 5 rows
    df_top5 = df.head(5)
    # Overwrite the original file (no index column)
    df_top5.to_excel(path, sheet_name=sheet_name, index=False)
    print(f"File {path} has been trimmed.")

if __name__ == "__main__":
    # Path to your file
    excel_path = input("Now either drag onto terminal or type path to Dataframe with the Filtered ESGF search results:")
    excel_path = Path(excel_path.strip(" "))  # strip needed as dragging onto terminal adds a trailing 'space'
    keep_top_five_rows(excel_path, sheet_name='Sheet1')


## Step 4. Fetch the data

The next step is to run the downloader script, which will download the filtered search results from the `ESGF_search_<varstamp>.xlsx` dataframe.

You can indicate where you'd like files to be downloaded to or keep `~/Desktop/search_results` created in **Step 2** as your default.

Downloads will trigger in parallel, and files will be organised under a directory tree that has a directory named `CMIP6` at the top.

Run the following cell:


In [ ]:
%run scripts/intake_OceanVarsDL.py


PS. This is equivalent to running the command `> python scripts/intake_OcanVarsDL.py` on a `Terminal prompt`.

When the program finishes running, a folder `CMIP6` should have been created within your `download_path` with the data organised per model.


## Step 5. Fetch Grid cell measures (`areacello` and `volcello`)

Run the next script to fetch corresponding grid cell measures `areacello` and `volcello` for the downloaded ESM outputs.

The program will import automatically the variables from *all* the available dataframes `DF_Downloadable_<varstamp>
.xlsx` residing in the parent directory that you indicate (if you want a specific dataframe to spawn the cell measure
search, then provide the full path to that specific `DF_Downloadable_<varstamp>.xlsx` dataframe).

Then the program will create a new dataframe `DF_Downloadable_<cellmeasure_stamp>.xlsx` on the chosen
`download_path`, with the cell measure search results.

Then you'll be prompted to indicate the path to this newly created dataframe `DF_Downloadable_<cellmeasure_stamp>.xlsx`, and the cell measure downloads will trigger in parallel.

Files will be organised under the same directory tree, with a directory `CMIP6` at the top.

Run the following:

In [ ]:
%run scripts/intake_CellMeasuresDL.py


PS. This is equivalent to running the command `> python scripts/intake_CellMeasuresDL.py` on a `Terminal prompt`.


## Step 6. Check files

That's it. Now inspect `download_path` to check the directory tree and if the files were downloaded.
